[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_model = d_model
        self.num_heads = num_heads

    def forward(self, x_q, x_kv):
        """
        x_q: bs, seq_q, d_model
        x_kv: bs, seq_kv, d_model
        """
        bs, seq_q, _ = x_q.shape
        _, seq_kv, _ = x_kv.shape
        d_k = self.d_model / self.num_heads
        Q = self.W_q(x_q).reshape(bs, seq_q, self.num_heads, -1).transpose(1,2) # bs, n_head, seq_q, d_k 
        K = self.W_k(x_kv).reshape(bs, seq_kv, self.num_heads, -1).transpose(1,2) # bs, n_head, seq_k, d_k 
        V = self.W_v(x_kv).reshape(bs, seq_kv, self.num_heads, -1).transpose(1,2) # bs, n_head, seq_k, d_k 
        scores = Q @ K.mT / math.sqrt(d_k) # bs, n_head, seq_q, seq_k
        att = torch.softmax(scores, -1) @ V # bs, n_head, seq_q, d_k 
        att = att.transpose(1, 2).reshape(bs, seq_q, -1)
        return self.W_o(att)

In [12]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.9ms)
  ✅ [2/4] Q and KV different lengths (1.0ms)
  ✅ [3/4] No causal mask — all KV affects all Q (43.3ms)
  ✅ [4/4] Gradient flow (24.9ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (71.1ms total)
  Progress saved. Run status() to see your dashboard.

